
# WildChat Analysis & MCP Prototype  
**Author:** Minsi Lu  
**Date:** Dec 2025  

This notebook serves two main purposes:  
1. **Data Insights** – Analyze the WildChat dataset to understand user behavior.  
2. **MCP Unit Tests** – Treat SQL queries and logic as unit tests for Model Context Protocol (MCP) server tools.  

---

### Dataset Overview  
We work with a cleaned subset of the [WildChat](https://huggingface.co/datasets/allenai/WildChat-4.8M)  dataset approximately **1.6M English conversations**.
- Removed rows with null values.  
- Added **precomputed B Tree indices** for fast queries:  
  ```python
  standard_indices = [
      ("idx_topic", "topic"),
      ("idx_model", "model_family"),
      ("idx_time", "timestamp"),
      ("idx_country", "country")
  ]
  ```
- Built **Full Text Search (FTS)** index on `search_text` for efficient keyword search:
  ```python
  PRAGMA create_fts_index('wildchat', 'id', 'search_text')
  ```
FTS index is an inverted index designed for efficient keyword-based text retrieval, and it can be combined with BM25 ranking to deliver highly relevant search results.
The cleaned dataset is available at [link]().

We explore the question we list in our proposal, which includes model preference, topic distribution and temporal trends, also, We attempt to address some questions that users might have. To enable fast analytics and search, we use DuckDB as the query engine. 

### Design Philosophy
“Macro-Analytics, Micro-Inspection”

Our system follows a three-tier information retrieval strategy:

- Macro-Analytics: High-level aggregation (trends, distributions).
- Discovery: Find specific conversations based on criteria (search, filtering).
- Micro-Inspection: Deep reading of individual conversation logs.

---

## Setup
We utilize DuckDB for high-performance OLAP queries on our processed dataset.

In [ ]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json

# Setup visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

DB_PATH = "data/wildchat.db" # TODO: Adjust path if necessary
con = duckdb.connect(DB_PATH, read_only=True)

print(f"Connected to {DB_PATH}")

ModuleNotFoundError: No module named 'duckdb'

## Meta-Analysis: Dataset Health & Overview
Before diving into specific trends, an agent (or a human analyst) needs to understand the "shape" of the data. This corresponds to the initialization Phase of an agentic workflow. At meta analysis level, all the tools are used for answer the question about the whole dataset, rather than a model / range of time.

### 1. Data summary
In a typical scenario, calculating distributions on 1.6M rows takes several seconds (or minutes depending on complexity). However, for an MCP Agent, latency is critical. An agent exploring a dataset shouldn't wait 10 seconds every time it asks "What topics are available?".

We implemented an ETL pipeline step `create_meta.py` that pre-computes aggregate statistics (Counts, Distributions, Date Ranges) and stores them in a dataset_meta table.
- Without Meta Table: `SELECT COUNT(*), ... FROM wildchat` -> Scans 1.6M rows -> O(N) Latency.
- With Meta Table: `SELECT * FROM dataset_meta` -> Reads 1 row -> O(1) Latency.

The function below demonstrates querying this pre-computed layer, serving as the unit test for our `get_dataset_summary` MCP tool.

MCP Tool Mapping:
- Tool: get_dataset_summary
- Example input:  Give me an overview of the dataset.

In [ ]:
# MCP Prototype: get_dataset_summary(summary_level='basic')

def get_db_connection():
    return duckdb.connect(DB_PATH, read_only=True)

def get_dataset_summary() -> str:
    """
    Retrieves high-level statistics about the WildChat dataset.
    Returns total counts, date ranges, and distributions for models, topics, and geography.
    Use this tool FIRST to understand the dataset's shape.
    """
    con = get_db_connection()
    try:
        query = """
            SELECT 
                total_count, 
                CAST(start_date AS VARCHAR) as start_date, 
                CAST(end_date AS VARCHAR) as end_date,
                model_json, 
                topic_json, 
                country_json 
            FROM dataset_meta
        """
        result = con.execute(query).fetchone()
        
        if not result:
            return "Error: Metadata not found. Please run the ingestion pipeline."

        total, start, end, model_str, topic_str, country_str = result
        
        stats = {
            "Total Conversations": total,
            "Date Range": f"{start} to {end}",
            "Top Models": json.loads(model_str),
            "Top Topics": json.loads(topic_str),
            "Top Countries": json.loads(country_str)
        }
        

    except Exception as e:
        return f"Error retrieving summary: {str(e)}"
    finally:
        con.close()

summary = get_dataset_summary()
for k, v in summary.items():
    print(f"- {k}: {v}")

### 2. Schema Awareness
For an LLM (like Claude) to generate correct SQL queries or perform accurate filtering, it must understand the database structure (table names, columns, and data types) without hallucination.

MCP Tool Mapping:
- Tool: get_db_schema
- User Intent: "What columns are available?" or "How is the data structured?"

In [ ]:

def get_db_schema() -> str:
    """
    Returns the schema of the 'wildchat' table.
    Useful for understanding available columns for filtering or deep analysis.
    """
    con = get_db_connection()
    try:
        df = con.execute("DESCRIBE wildchat").fetchdf()
        
        schema_info = []
        for _, row in df.iterrows():
            schema_info.append(f"- {row['column_name']} ({row['column_type']})")
            
        return "Table 'wildchat' Schema:\n" + "\n".join(schema_info)
    except Exception as e:
        return f"Error getting schema: {str(e)}"
    finally:
        con.close()
schema = get_db_schema()
print(schema)

## Topic Distribution Analysis
这一部分

Understanding what users discuss is crucial for categorizing the dataset. We utilized BERTopic (with UMAP/HDBSCAN) to cluster conversations into semantic topics.

Research Question: What are the most common themes users discuss with LLMs?

Are they mostly technical (Coding)?

Creative (Writing)?

Or personal (Roleplay/Advice)?

MCP Tool Mapping:

Tool: get_topic_distribution / generate_insights_prompt

User Intent: "What are the most popular topics?"

Implementation: Aggregation on the topic column (indexed).

In [ ]:
# Analysis: Visualizing Topic Distribution
def analyze_topic_distribution():
    query = """
        SELECT topic, COUNT(*) as count
        FROM wildchat
        WHERE topic != 'General / Noise' AND topic IS NOT NULL
        GROUP BY topic
        ORDER BY count DESC
        LIMIT 20
    """
    df = con.execute(query).fetchdf()
    
    # Visualization
    plt.figure(figsize=(12, 8))
    sns.barplot(data=df, x='count', y='topic', palette='viridis')
    plt.title('Top 20 Conversation Topics (BERTopic Clustered)')
    plt.xlabel('Number of Conversations')
    plt.ylabel('Topic Category')
    plt.show()
    
    return df

top_topics = analyze_topic_distribution()
print(top_topics.head())

4. Model Comparison & Preferences
The dataset spans multiple versions of GPT models. Analyzing the distribution reveals user preference or availability during the collection period.

Research Question: Which models generate the most engagement? Are there shifts in model usage?

MCP Tool Mapping:

Tool: get_dataset_summary (Detailed view)

Implementation: Group by model_family.

In [ ]:
def analyze_model_distribution():
    query = """
        SELECT model_family, COUNT(*) as count
        FROM wildchat
        GROUP BY model_family
        ORDER BY count DESC
    """
    df = con.execute(query).fetchdf()
    
    # Pie Chart for Model Families
    plt.figure(figsize=(8, 8))
    plt.pie(df['count'], labels=df['model_family'], autopct='%1.1f%%', startangle=140)
    plt.title('Distribution of Model Families')
    plt.show()

analyze_model_distribution()

Temporal Trends
Static counts don't tell the whole story. We need to see how interaction volume evolves over time, potentially correlating with major events (e.g., the release of GPT-4).

Research Question: How has user interaction evolved over time?

MCP Tool Mapping:

Tool: analyze_trends (Planned)

User Intent: "Show me the activity trend in 2024."

Implementation: Time-series aggregation using date_trunc

In [ ]:
def analyze_temporal_trends():
    # Aggregating by Month
    query = """
        SELECT 
            date_trunc('month', timestamp) as month,
            COUNT(*) as conversations
        FROM wildchat
        GROUP BY month
        ORDER BY month
    """
    df = con.execute(query).fetchdf()
    
    # Line Plot
    plt.figure(figsize=(14, 6))
    sns.lineplot(data=df, x='month', y='conversations', marker='o', linewidth=2.5)
    plt.title('Monthly Conversation Volume')
    plt.xlabel('Date')
    plt.ylabel('Volume')
    plt.grid(True)
    plt.show()

analyze_temporal_trends()

6. Engagement Analysis: Length & Depth
Do certain topics trigger longer conversations? We use token_count (estimated during ingestion) and turn_count as proxies for engagement.

Research Question: Are certain topics associated with longer or more interactive dialogues?

MCP Tool Mapping:

Tool: analyze_engagement

User Intent: "Which topics require the most tokens?"

In [ ]:
def analyze_engagement_by_topic():
    # We look at the average token count per topic
    query = """
        SELECT 
            topic, 
            AVG(token_count) as avg_tokens,
            AVG(turn_count) as avg_turns
        FROM wildchat
        WHERE topic != 'General / Noise'
        GROUP BY topic
        HAVING COUNT(*) > 1000 -- Filter out rare topics for stability
        ORDER BY avg_tokens DESC
        LIMIT 15
    """
    df = con.execute(query).fetchdf()
    
    # Visualization
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df, x='avg_tokens', y='topic', palette='magma')
    plt.title('Average Conversation Length (Tokens) by Topic')
    plt.xlabel('Avg Tokens')
    plt.show()

analyze_engagement_by_topic()

Discovery Layer: Semantic Search (BM25)
This section prototypes the core Discovery capability. We move from aggregate statistics to finding specific needles in the haystack. We utilize DuckDB's Full Text Search (FTS) extension.

Research Question: Can we efficiently retrieve conversations about specific technical concepts (e.g., "Segmentation Fault" or "Quantum Physics")?

MCP Tool Mapping:

Tool: search_conversations

User Intent: "Find examples of users asking about Python memory leaks."

Implementation: fts_main_wildchat.match_bm25(id, query)

In [ ]:
# MCP Prototype: search_conversations(query, limit)
def test_search_tool(query, limit=3):
    print(f"🔎 Searching for: '{query}'")
    
    # Note: We rely on the FTS index created in the pipeline
    sql = f"""
        SELECT 
            id, 
            topic, 
            substr(search_text, 1, 150) as preview, 
            fts_main_wildchat.match_bm25(id, '{query}') AS score
        FROM wildchat
        WHERE score IS NOT NULL
        ORDER BY score DESC
        LIMIT {limit}
    """
    
    results = con.execute(sql).fetchdf()
    
    if results.empty:
        print("No results found.")
        return

    for idx, row in results.iterrows():
        print(f"--- Result {idx+1} (Score: {row['score']:.2f}) ---")
        print(f"ID: {row['id']}")
        print(f"Topic: {row['topic']}")
        print(f"Preview: {row['preview']}...")
        print("")

# Test Cases
test_search_tool("segmentation fault c++")
test_search_tool("love story romeo juliet")

8. Micro-Inspection: Resource Retrieval
Finally, once a relevant conversation ID is found via Search, the agent needs to retrieve the full content. This corresponds to the MCP Resource primitive.

MCP Resource Mapping:

Resource URI: wildchat://conversations/{id}

Implementation: Retrieve the full_content JSON blob

In [ ]:
# MCP Prototype: Read Resource wildchat://conversations/{id}
def test_get_conversation_resource(conv_id):
    query = "SELECT full_content FROM wildchat WHERE id = ?"
    result = con.execute(query, [conv_id]).fetchone()
    
    if not result:
        print("Resource not found.")
        return
    
    # The DB stores it as a JSON string, we parse it back to Python dict
    full_log = json.loads(result[0])
    
    print(f"📖 Reading Conversation Resource: {conv_id}")
    print(f"Total Turns: {len(full_log)}")
    print("-" * 40)
    
    # Display first 2 turns for brevity
    for turn in full_log[:2]:
        role = turn['role'].upper()
        content = turn['content'][:200].replace('\n', ' ') # Truncate for display
        print(f"[{role}]: {content}...")

# Use an ID found in the previous search step (hardcoded example for demo)
# You can replace this with an ID from your DB
sample_id = con.execute("SELECT id FROM wildchat LIMIT 1").fetchone()[0]
test_get_conversation_resource(sample_id)